# Difference-in-Differences: Applications

**P@S 2026, Lecture 12 (Mar 2)**

On Wednesday we built the DiD toolkit (2x2 table, TWFE, event study). Today we apply it to a new setting inspired by Kim and Kam (2023), who measured anti-Chinese bias in Yelp restaurant reviews during the COVID-19 pandemic.

**What we'll do:**
1. Simulate a dataset inspired by Kim & Kam (2023)
2. Visualize parallel trends and the treatment effect
3. Run DiD by hand and by regression
4. Specificity test: are other cuisines affected?
5. Event study: when does the effect appear?

## Part 1: Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
np.random.seed(42)

## Part 2: Simulate the Yelp ratings data

We simulate a dataset inspired by Kim and Kam (2023). The key elements:
- **Two types of restaurants**: Chinese and American
- **Time period**: 24 months (Jan 2019 - Dec 2020)
- **Treatment shock**: COVID-19 lockdown (March 2020, month 15)
- **Treatment effect**: Chinese restaurants receive lower ratings after COVID, American restaurants don't

We simulate 200 restaurants (100 Chinese, 100 American) across 8 metro areas, each receiving monthly review ratings.

In [ ]:
# Simulate restaurant-month panel data
n_restaurants = 200
n_months = 24
lockdown_month = 15  # March 2020

# Restaurant attributes
restaurant_ids = np.arange(n_restaurants)
cuisine = np.array(['Chinese'] * 100 + ['American'] * 100)
metro = np.random.choice(['NYC', 'LA', 'Chicago', 'Houston', 'Phoenix', 'Philly', 'Dallas', 'SF'],
                         size=n_restaurants)

# Restaurant-level baseline rating (random intercept)
base_rating = np.random.normal(3.9, 0.3, n_restaurants)

# Build panel
rows = []
for i in range(n_restaurants):
    for m in range(1, n_months + 1):
        # Common time trend: slight upward drift for all restaurants
        time_trend = 0.005 * m

        # Treatment effect: Chinese restaurants get lower ratings after lockdown
        treat_effect = 0.0
        if cuisine[i] == 'Chinese' and m >= lockdown_month:
            treat_effect = -0.15  # About 0.15 stars lower

        # General COVID bump (American restaurants got slightly better reviews post-lockdown)
        covid_bump = 0.08 if m >= lockdown_month else 0.0

        # Rating with noise
        rating = base_rating[i] + time_trend + treat_effect + covid_bump + np.random.normal(0, 0.25)
        rating = np.clip(rating, 1, 5)

        rows.append({
            'restaurant_id': i,
            'cuisine': cuisine[i],
            'metro': metro[i],
            'month': m,
            'post_lockdown': int(m >= lockdown_month),
            'chinese': int(cuisine[i] == 'Chinese'),
            'rating': round(rating, 2)
        })

df = pd.DataFrame(rows)
print(f"Dataset: {len(df)} restaurant-month observations")
print(f"Restaurants: {df['restaurant_id'].nunique()} ({df[df['chinese']==1]['restaurant_id'].nunique()} Chinese, {df[df['chinese']==0]['restaurant_id'].nunique()} American)")
print(f"Months: {df['month'].nunique()} (lockdown at month {lockdown_month})")
df.head()

## Part 3: Visualize the data

Plot average ratings over time for Chinese and American restaurants. This is the analogue of Kim and Kam's Figure 1.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Average ratings by cuisine and month
for cuisine_type, color, marker in [('Chinese', '#c0392b', 'o'), ('American', '#2980b9', 's')]:
    group = df[df['cuisine'] == cuisine_type].groupby('month')['rating'].mean()
    ax.plot(group.index, group.values, f'{marker}-', color=color, linewidth=2,
            markersize=6, label=f'{cuisine_type} restaurants')

# Lockdown line
ax.axvline(x=lockdown_month, color='gray', linestyle='--', linewidth=2, alpha=0.7)
ax.text(lockdown_month + 0.3, ax.get_ylim()[0] + 0.01, 'COVID-19\nlockdown',
        fontsize=11, color='gray', va='bottom')

# Month labels
month_labels = ['Jan 19', '', 'Mar', '', 'May', '', 'Jul', '', 'Sep', '', 'Nov', '',
                'Jan 20', '', 'Mar', '', 'May', '', 'Jul', '', 'Sep', '', 'Nov', 'Dec']
ax.set_xticks(range(1, 25))
ax.set_xticklabels(month_labels, fontsize=8, rotation=45)
ax.set_ylabel('Average Yelp Rating', fontsize=12)
ax.set_title('Yelp Ratings: Chinese vs. American Restaurants (Simulated)', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Before lockdown, ratings for both cuisines track together (parallel trends).")
print("After lockdown, Chinese restaurant ratings drop while American ratings stay stable or rise.")

## Part 4: DiD by hand (the 2x2 table)

Same arithmetic from Wednesday, new context.

In [ ]:
# 2x2 table
cn_pre  = df[(df['chinese'] == 1) & (df['post_lockdown'] == 0)]['rating'].mean()
cn_post = df[(df['chinese'] == 1) & (df['post_lockdown'] == 1)]['rating'].mean()
am_pre  = df[(df['chinese'] == 0) & (df['post_lockdown'] == 0)]['rating'].mean()
am_post = df[(df['chinese'] == 0) & (df['post_lockdown'] == 1)]['rating'].mean()

print("              Pre-lockdown   Post-lockdown   Change")
print(f"Chinese:      {cn_pre:.3f}         {cn_post:.3f}          {cn_post - cn_pre:+.3f}")
print(f"American:     {am_pre:.3f}         {am_post:.3f}          {am_post - am_pre:+.3f}")
print(f"\nDiD estimate: ({cn_post - cn_pre:.3f}) - ({am_post - am_pre:.3f}) = {(cn_post - cn_pre) - (am_post - am_pre):.3f}")
print(f"\nChinese restaurants received ratings about {abs((cn_post - cn_pre) - (am_post - am_pre)):.2f} stars lower")
print("after COVID, relative to American restaurants.")

## Part 5: TWFE regression

The regression version: restaurant fixed effects + month fixed effects + treatment indicator.

In [ ]:
# Create treatment variable: Chinese x PostLockdown
df['treated'] = df['chinese'] * df['post_lockdown']

# TWFE with restaurant and month fixed effects, clustered by restaurant
model = smf.ols('rating ~ treated + C(restaurant_id) + C(month)', data=df).fit(
    cov_type='cluster', cov_kwds={'groups': df['restaurant_id']}
)

print("TWFE Regression: rating ~ treated + restaurant FE + month FE")
print(f"  DiD estimate (treated):  {model.params['treated']:.4f}")
print(f"  Clustered SE:            {model.bse['treated']:.4f}")
print(f"  t-statistic:             {model.tvalues['treated']:.2f}")
print(f"  p-value:                 {model.pvalues['treated']:.6f}")
print(f"\nThe regression recovers the treatment effect: Chinese restaurants")
print(f"received ratings about {abs(model.params['treated']):.2f} stars lower after the lockdown.")

## Part 6: The interaction form

Writing it out explicitly, as in the slides:

$$Y_{it} = \alpha + \beta_1 \cdot \text{Chinese} + \beta_2 \cdot \text{PostLockdown} + \beta_3 \cdot (\text{Chinese} \times \text{PostLockdown}) + \varepsilon_{it}$$

$\beta_3$ is the DiD estimate.

In [ ]:
# Interaction form
model_int = smf.ols('rating ~ chinese * post_lockdown', data=df).fit(
    cov_type='cluster', cov_kwds={'groups': df['restaurant_id']}
)

print("Interaction form: rating ~ Chinese + PostLockdown + Chinese:PostLockdown\n")
print(model_int.summary().tables[1])
print(f"\nMapping to the 2x2 table:")
print(f"  Intercept (American, pre):      {model_int.params['Intercept']:.3f}")
print(f"  Chinese (level difference):     {model_int.params['chinese']:+.3f}")
print(f"  PostLockdown (common trend):    {model_int.params['post_lockdown']:+.3f}")
print(f"  Chinese:PostLockdown (DiD):     {model_int.params['chinese:post_lockdown']:+.3f}")

## Part 7: Specificity test

Kim and Kam's key insight: if this is really anti-Chinese bias (not general pandemic disruption), then other Asian cuisines should NOT be affected. Let's add Japanese restaurants to our simulation and test.

In [ ]:
# Add Japanese, Korean, and Thai restaurants (no treatment effect)
extra_cuisines = ['Japanese', 'Korean', 'Thai']
extra_rows = []
for cuisine_name in extra_cuisines:
    for i in range(50):  # 50 restaurants per cuisine
        rid = n_restaurants + i + extra_cuisines.index(cuisine_name) * 50
        base = np.random.normal(3.9, 0.3)
        for m in range(1, n_months + 1):
            time_trend = 0.005 * m
            covid_bump = 0.08 if m >= lockdown_month else 0.0
            # NO anti-cuisine treatment effect for these cuisines
            rating = base + time_trend + covid_bump + np.random.normal(0, 0.25)
            rating = np.clip(rating, 1, 5)
            extra_rows.append({
                'restaurant_id': rid,
                'cuisine': cuisine_name,
                'metro': np.random.choice(['NYC', 'LA', 'Chicago', 'Houston', 'Phoenix', 'Philly', 'Dallas', 'SF']),
                'month': m,
                'post_lockdown': int(m >= lockdown_month),
                'chinese': 0,
                'rating': round(rating, 2),
                'treated': 0
            })

df_all = pd.concat([df, pd.DataFrame(extra_rows)], ignore_index=True)

# Run DiD for each cuisine vs American
print("Specificity test: DiD estimate for each Asian cuisine vs. American restaurants\n")
print(f"{'Cuisine':<12} {'DiD estimate':>12}  {'SE':>8}  {'p-value':>8}")
print("-" * 45)

for cuisine_name in ['Chinese', 'Japanese', 'Korean', 'Thai']:
    sub = df_all[df_all['cuisine'].isin([cuisine_name, 'American'])].copy()
    sub['is_asian'] = (sub['cuisine'] == cuisine_name).astype(int)
    sub['treat'] = sub['is_asian'] * sub['post_lockdown']
    m = smf.ols('rating ~ is_asian * post_lockdown', data=sub).fit(
        cov_type='cluster', cov_kwds={'groups': sub['restaurant_id']}
    )
    coef = m.params['is_asian:post_lockdown']
    se = m.bse['is_asian:post_lockdown']
    pval = m.pvalues['is_asian:post_lockdown']
    sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''
    print(f"{cuisine_name:<12} {coef:>+12.4f}  {se:>8.4f}  {pval:>8.4f} {sig}")

print("\nOnly Chinese restaurants show a significant negative effect.")
print("This specificity supports the 'othering' interpretation.")

## Part 8: Event study

Let the treatment effect vary by month. Pre-treatment coefficients should be near zero (parallel trends test); post-treatment coefficients show the effect dynamics.

In [ ]:
# Event study: interact Chinese with each month (omit month 14 = Feb 2020 as reference)
df['rel_month'] = df['month'] - lockdown_month  # 0 = lockdown month

# Create period dummies (quarterly bins for cleaner plot)
df['quarter'] = pd.cut(df['rel_month'],
                       bins=[-15, -9, -6, -3, -1, 0, 3, 6, 10],
                       labels=['Q-4', 'Q-3', 'Q-2', 'Q-1(ref)', 'Q0', 'Q+1', 'Q+2', 'Q+3'])

# Use month-level interactions for a proper event study
ref_month = 14  # Feb 2020, one month before lockdown
event_months = [m for m in range(1, 25) if m != ref_month]

formula_parts = []
for m in event_months:
    col = f'cn_m{m}'
    df[col] = ((df['month'] == m) & (df['chinese'] == 1)).astype(int)
    formula_parts.append(col)

formula = 'rating ~ ' + ' + '.join(formula_parts) + ' + C(restaurant_id) + C(month)'

# Fit (this may take a moment with 200 restaurant fixed effects)
event_model = smf.ols(formula, data=df).fit(
    cov_type='cluster', cov_kwds={'groups': df['restaurant_id']}
)

# Extract coefficients
months_plot = list(range(1, 25))
coefs = []
ses = []
for m in months_plot:
    col = f'cn_m{m}'
    if m == ref_month:
        coefs.append(0)
        ses.append(0)
    else:
        coefs.append(event_model.params[col])
        ses.append(event_model.bse[col])

# Plot
fig, ax = plt.subplots(figsize=(14, 6))
rel_months = [m - lockdown_month for m in months_plot]

ax.errorbar(rel_months, coefs, yerr=[1.96 * s for s in ses],
            fmt='o', color='#2c3e50', markersize=5, capsize=3, capthick=1.5, linewidth=1)
ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.8)
ax.axvline(x=0, color='#c0392b', linestyle='--', linewidth=2, alpha=0.7)

ax.fill_between([-15, -0.5], -0.5, 0.5, alpha=0.03, color='blue')
ax.fill_between([-0.5, 10], -0.5, 0.5, alpha=0.03, color='red')
ax.text(-7, 0.35, 'Pre-lockdown\n(should be near 0)', fontsize=11, color='#2980b9', ha='center')
ax.text(5, 0.35, 'Post-lockdown\n(treatment effect)', fontsize=11, color='#c0392b', ha='center')

ax.set_xlabel('Months relative to COVID lockdown (March 2020 = 0)', fontsize=12)
ax.set_ylabel('Estimated coefficient (Chinese x month)', fontsize=12)
ax.set_title('Event Study: Anti-Chinese Bias in Yelp Ratings', fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Pre-lockdown coefficients cluster around zero (parallel trends holds).")
print("Post-lockdown coefficients are negative (Chinese restaurants penalized).")

## Part 9: Summary

| Step | Wednesday (Concepts) | Today (Applications) |
|------|---------------------|---------------------|
| **Data** | Organ donation (real) | Yelp ratings (simulated, inspired by Kim & Kam) |
| **Treated** | California (active choice) | Chinese restaurants |
| **Control** | 25 other states | American restaurants |
| **DiD estimate** | -0.022 (reduced donation) | ~-0.15 stars (lower ratings) |
| **Specificity** | N/A | Only Chinese, not Japanese/Korean/Thai |
| **Event study** | Flat pre, negative post | Flat pre, negative post |

**The same DiD toolkit works across domains.** The method doesn't care whether the outcome is organ donation rates or Yelp stars. What matters is: (1) a clear treatment shock, (2) a plausible control group, (3) parallel trends before treatment.

## Part 10: The big picture — it's all OLS

Every causal inference method we've seen in this course uses OLS (ordinary least squares) as the computational engine. What changes is the **identification strategy**: the argument for why $\hat{\beta}$ estimates a causal effect.

| Method | Identification strategy | The regression | Course example |
|:-------|:----------------------|:---------------|:---------------|
| **Randomized experiment** | Random assignment breaks confounding | $Y = \alpha + \beta T + \varepsilon$ | Allcott (2020): Facebook deactivation |
| **Difference-in-differences** | Parallel trends + treatment timing | $Y = \alpha + \gamma_{\text{group}} + \delta_{\text{time}} + \beta(\text{Group} \times \text{Post}) + \varepsilon$ | Kim & Kam (2023): Yelp ratings |
| **Regression discontinuity** | Continuity at a sharp cutoff | $Y = \alpha + \beta T + f(X) + \varepsilon$ | *(next week)* |

**The punchline:** $\beta$ is always estimated the same way — minimize $\sum (Y_i - \hat{Y}_i)^2$. The "method" is the reason you believe $\beta$ is causal, not just correlational.

In [ ]:
# Part 10 demo: all three methods use sm.OLS under the hood

# --- Method 1: Randomized experiment (simple treatment indicator) ---
# Y = alpha + beta * T + epsilon
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

np.random.seed(123)
n = 500
T_exp = np.random.binomial(1, 0.5, n)
Y_exp = 2.0 + 0.5 * T_exp + np.random.normal(0, 1, n)
X_exp = sm.add_constant(T_exp)
m1 = sm.OLS(Y_exp, X_exp).fit()

# --- Method 2: Difference-in-differences (from our simulation above) ---
# Y = alpha + gamma_group + delta_time + beta * (Group x Post) + epsilon
m2 = smf.ols('rating ~ chinese * post_lockdown', data=df).fit()

# --- Method 3: Regression discontinuity (simulated) ---
# Y = alpha + beta * T + f(X) + epsilon
np.random.seed(456)
X_rd = np.random.uniform(-1, 1, 1000)
T_rd = (X_rd >= 0).astype(int)
Y_rd = 1.0 + 0.8 * T_rd + 2.0 * X_rd + np.random.normal(0, 0.5, 1000)
X_mat = np.column_stack([np.ones(1000), T_rd, X_rd])
m3 = sm.OLS(Y_rd, X_mat).fit()

# Print the key coefficient from each
print("Method                    beta_hat    SE        Engine")
print("-" * 60)
print(f"Randomized experiment     {m1.params[1]:+.4f}    {m1.bse[1]:.4f}    sm.OLS")
print(f"Diff-in-differences       {m2.params['chinese:post_lockdown']:+.4f}    {m2.bse['chinese:post_lockdown']:.4f}    smf.ols (= sm.OLS)")
print(f"Regression discontinuity  {m3.params[1]:+.4f}    {m3.bse[1]:.4f}    sm.OLS")
print()
print("Same function. Different identification arguments.")